In [4]:
import tensorflow as tf
from tensorflow.keras.layers import Dense, Flatten, Conv2D, MaxPooling2D


In [5]:
from tensorflow.keras.models import Sequential

In [9]:
import cv2
import numpy as np
import os
import imghdr

C:\Users\gurud\AppData\Local\Temp\ipykernel_30108\1771036855.py:4: DeprecationWarning: 'imghdr' is deprecated and slated for removal in Python 3.13
  import imghdr


In [8]:
data_dir="data"

In [10]:
exts=["jpg","jpeg","png"]
for image_class in os.listdir(data_dir):
    class_dir = os.path.join(data_dir, image_class)
    for image_name in os.listdir(class_dir):
        image_path = os.path.join(class_dir, image_name)
        image = cv2.imread(image_path)
        tip=imghdr.what(image_path)
        if tip not in exts:
            print("Deleted file: ",image_path)
            os.remove(image_path)

In [11]:
data=tf.keras.utils.image_dataset_from_directory(
    data_dir)

Found 3352 files belonging to 7 classes.


In [12]:
model=Sequential()

In [13]:
model.add(Conv2D(16,(3,3),activation='relu',input_shape=(256,256,3)))
model.add(MaxPooling2D((2,2)))
model.add(Conv2D(32,(3,3),activation='relu'))
model.add(MaxPooling2D((2,2)))
model.add(Flatten())
model.add(Dense(128,activation='relu'))
model.add(Dense(7,activation='softmax'))

c:\Users\gurud\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [14]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 254, 254, 16)   │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 127, 127, 16)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 125, 125, 32)   │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 62, 62, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 123008)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    15,745,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,751,143 (60.09 MB)

 Trainable params: 15,751,143 (60.09 MB)

 Non-trainable params: 0 (0.00 B)

In [15]:
len(data)

105

In [16]:
train_size=80
test_size=15
val_size=10

In [17]:
data_it=data.as_numpy_iterator()
data_it.next()

(array([[[[1.68000000e+02, 1.87000000e+02, 1.91000000e+02],
          [1.68000000e+02, 1.87000000e+02, 1.91000000e+02],
          [1.68000000e+02, 1.87000000e+02, 1.91000000e+02],
          ...,
          [1.82148438e+02, 1.95148438e+02, 2.01148438e+02],
          [1.75546875e+02, 1.89289062e+02, 1.93062500e+02],
          [1.68515625e+02, 1.82515625e+02, 1.85515625e+02]],
 
         [[1.68000000e+02, 1.87000000e+02, 1.91000000e+02],
          [1.68000000e+02, 1.87000000e+02, 1.91000000e+02],
          [1.68000000e+02, 1.87000000e+02, 1.91000000e+02],
          ...,
          [1.80627197e+02, 1.93627197e+02, 1.99627197e+02],
          [1.71811890e+02, 1.85554077e+02, 1.89327515e+02],
          [1.61499756e+02, 1.75499756e+02, 1.78499756e+02]],
 
         [[1.68140625e+02, 1.87000000e+02, 1.91000000e+02],
          [1.68140625e+02, 1.87000000e+02, 1.91000000e+02],
          [1.68140625e+02, 1.87000000e+02, 1.91000000e+02],
          ...,
          [1.79248413e+02, 1.92389038e+02, 1.9796

In [18]:
data=data.map(lambda x,y: (x/255,y))

In [19]:
train=data.take(train_size)
test=data.skip(train_size).take(test_size)
val=data.skip(train_size+test_size).take(val_size)

In [20]:
model.compile("adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

In [21]:
model.fit(train,epochs=5,validation_data=val)

Epoch 1/5
80/80 ━━━━━━━━━━━━━━━━━━━━ 22s 264ms/step - accuracy: 0.3086 - loss: 2.0280 - val_accuracy: 0.4263 - val_loss: 1.5992
Epoch 2/5
80/80 ━━━━━━━━━━━━━━━━━━━━ 22s 267ms/step - accuracy: 0.6699 - loss: 1.0012 - val_accuracy: 0.5353 - val_loss: 1.3634
Epoch 3/5
80/80 ━━━━━━━━━━━━━━━━━━━━ 21s 260ms/step - accuracy: 0.9059 - loss: 0.3345 - val_accuracy: 0.5769 - val_loss: 1.3523
Epoch 4/5
80/80 ━━━━━━━━━━━━━━━━━━━━ 21s 261ms/step - accuracy: 0.9793 - loss: 0.0999 - val_accuracy: 0.6474 - val_loss: 1.4150
Epoch 5/5
80/80 ━━━━━━━━━━━━━━━━━━━━ 21s 262ms/step - accuracy: 0.9934 - loss: 0.0480 - val_accuracy: 0.5994 - val_loss: 1.7323


In [22]:
model.save("cars.h5")